# Abstract Reviewer (GPT-5 mini)

This notebook generates a conference-style abstract review rationale constrained to a target score, then returns project JSON fields: `PAPER_ID`, `SCORE`, `RATIONAL`.

In [ ]:
# If needed:
# %pip install -q openai

import os
import json
import re
from typing import List, Dict, Any
from openai import OpenAI

# Set your key in environment before running:
# Windows PowerShell: setx OPENAI_API_KEY "your_key_here"
# or for current session: $env:OPENAI_API_KEY="your_key_here"

client = OpenAI()


In [ ]:
def _extract_json_obj(text: str) -> Dict[str, Any]:
    """Extract first JSON object from model text output."""
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    m = re.search(r'\{[\s\S]*\}', text)
    if not m:
        raise ValueError(f"Could not parse JSON from model output: {text[:300]}")
    obj = json.loads(m.group(0))
    if not isinstance(obj, dict):
        raise ValueError("Parsed JSON is not an object")
    return obj


def score_one(item: Dict[str, str], model: str = "gpt-5-mini") -> Dict[str, Any]:
    """
    item keys:
      PAPER_ID, title, submission, degradation_types, target_score, reviewer_style
    returns project format:
      {PAPER_ID, SCORE, RATIONAL}
    """
    paper_id = item["PAPER_ID"]
    title = item["title"]
    submission = item["submission"]
    degradation_types = item.get("degradation_types", "")
    target_score = int(item["target_score"])
    reviewer_style = item.get("reviewer_style", "balanced and concise")

    prompt = f"""
You are a conference reviewer evaluating only the abstract, not the full paper.

Task:
Evaluate the quality of the following research abstract for conference acceptance.

Reference:
A strong research abstract clearly presents:
- the research problem,
- the proposed methodology,
- the main contribution,
- and convincing experimental evidence.

Scoring Rubric:
0 = Strong reject
1 = Reject
2 = Borderline
3 = Accept
4 = Strong accept

Title:
{title}

Submitted abstract:
{submission}

Known abstract-level flaw profile:
{degradation_types}

Target score:
{target_score}


Instructions: 
1. Write a realistic reviewer rationale that matches the target score. 
2. Evaluate only the abstract quality, not the full paper. 
3. Mention both strengths and weaknesses visible in the abstract. 
4. Discuss concrete issues actually observable in the submission. 
5. Do not mention the original abstract. 
6. Do not mention synthetic data generation or degradation. 
7. Keep the rationale between 2 and 5 sentences IMPORTANT MUST FOLLOW THIS SIZE AS NEEDED . 
8. Maintain a professional conference-review tone. 
9. Include at least one positive aspect of the abstract. 
10. Clearly explain what should be improved or clarified.

Return valid JSON only in the following format:
{{
  "score": {target_score},
  "rationale": "..."
}}
"""

    resp = client.responses.create(
        model=model,
        input=prompt,
        temperature=0
    )

    text = getattr(resp, "output_text", "") or ""
    parsed = _extract_json_obj(text)

    score = int(parsed.get("score", target_score))
    score = max(0, min(4, score))
    rationale = str(parsed.get("rationale", "")).strip()

    return {
        "PAPER_ID": str(paper_id),
        "SCORE": score,
        "RATIONAL": rationale
    }


def score_batch(items: List[Dict[str, str]], model: str = "gpt-5-mini") -> List[Dict[str, Any]]:
    results = []
    for i, item in enumerate(items, 1):
        out = score_one(item, model=model)
        results.append(out)
        print(f"Scored {i}/{len(items)} -> {out['PAPER_ID']} : {out['SCORE']}")
    return results


In [ ]:
# Example input (replace with your real data)
items = [
    {
        "PAPER_ID": "paper_001",
        "title": "Example Paper Title",
        "submission": "This paper proposes ...",
        "degradation_types": "missing evidence, vague method details",
        "target_score": 2,
        "reviewer_style": "critical but constructive"
    },
    # Add more items here
]
import json

input_path = "data/abstracts.json"
with open(input_path, "r", encoding="utf-8") as f:
    items = json.load(f)
    
results = score_batch(items, model="gpt-5-mini")
print(json.dumps(results, ensure_ascii=False, indent=2))


In [ ]:
# Optional: save results to file
out_path = "scores.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Saved: {out_path}")
